# Comparative analysis of Forecasting Models

**Models:** Multiple linear regression, Polynomial regression, Ridge, Lasso, Elastic net, Random forest and XGBoost

In [5]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from xgboost import XGBRegressor


In [6]:
DF_PATH = r"..\data\processed\singapore\dataframe.parquet"
TARGET_COL = "system_demand_mw"

HORIZON = "1D"          # how far ahead each row is forecast
BASE_LAG = "lag_7D"     # reference for the "change vs last week" formulation; must be >= HORIZON

N_SPLITS = 4            # cross-validation folds
TEST_DAYS = 182         # length of each fold's test window, in days

TUNE = True             # Toggle hyperparameter tuning on or off
TUNE_SCOPE = "once"     # "once": tune on fold 1's training window, reuse everywhere (fast)
                        
INNER_SPLITS = 3        # time-series folds inside the tuning search
N_ITER_RF = 8           # random-search candidates for random forest
N_ITER_XGB = 12         # random-search candidates for XGBoost

RANDOM_STATE = 42

In [7]:
feats = pd.read_parquet(DF_PATH).sort_index()

gaps = feats.index.to_series().diff().dropna()
assert gaps.nunique() == 1, "Index is not a regular grid; the lag/rolling shifts below assume one."
step = gaps.iloc[0]
per_day = int(pd.Timedelta("1D") / step)
h_steps = int(pd.Timedelta(HORIZON) / step)
assert h_steps >= 1, "HORIZON must be at least one sampling interval."

print(f"{feats.shape[0]} rows x {feats.shape[1]} columns")
print(f"Horizon = {HORIZON} = {h_steps} steps")

69888 rows x 34 columns
Horizon = 1D = 48 steps


In [8]:
def duration_of(col):
    # "lag_1D" -> 1 day
    return pd.Timedelta(col.split("_")[-1])

lag_all  = [c for c in feats.columns if c.startswith("lag_")]
roll_cols = [c for c in feats.columns if c.startswith("roll_")]

lag_keep = [c for c in lag_all if duration_of(c) >= pd.Timedelta(HORIZON)]
lag_drop = [c for c in lag_all if c not in lag_keep]

hs = feats.copy()
if h_steps > 1:
    hs[roll_cols] = hs[roll_cols].shift(h_steps - 1)

if BASE_LAG not in lag_keep:
    raise ValueError(f"BASE_LAG '{BASE_LAG}' is not available at horizon {HORIZON}. Available lags: {lag_keep}")

## Separating features for different model families

- For linear models, we use the cyclical features (sin/cos) and one-hot encoded day and hour features
- Tree based models will use the raw datetime features like hours, time_of_day, etc.

In [15]:
cal_tree = [c for c in ["time_of_day", "hour", "day_of_week", "is_weekend", "day_of_month",
                        "day_of_year", "week_of_year", "month", "quarter"] if c in hs.columns]
cal_cyc  = [c for c in hs.columns if c.endswith(("_sin", "_cos"))]

tree_cols = cal_tree + lag_keep + roll_cols
lin_num   = cal_cyc + ["is_weekend"] + lag_keep + roll_cols
lin_cat   = ["day_of_week", "hour"]

# core features expanded by the polynomial model
poly_num = ["time_of_day", "is_weekend"] + lag_keep + [c for c in roll_cols if "_mean_" in c]
rest_num = [c for c in lin_num if c not in poly_num]

used = list(dict.fromkeys(tree_cols + lin_num + lin_cat + poly_num + [BASE_LAG]))

X = hs[used]
y = hs[TARGET_COL]

# drop rows where any feature or the target is missing
valid = X.notna().all(axis=1) & y.notna()
print(f"Dropping {int((~valid).sum())} rows with missing features/target "+f"({(~valid).mean() * 100:.2f}%)")
X, y = X[valid], y[valid]

print(f"\nTree features ({len(tree_cols)}): {tree_cols}")
print(f"\nLinear numeric ({len(lin_num)}), categorical: {lin_cat}")
print(f"\nPolynomial core ({len(poly_num)}): {poly_num}")

Dropping 383 rows with missing features/target (0.55%)

Tree features (23): ['time_of_day', 'hour', 'day_of_week', 'is_weekend', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'lag_1D', 'lag_7D', 'roll_mean_3h', 'roll_std_3h', 'roll_max_3h', 'roll_min_3h', 'roll_mean_1D', 'roll_std_1D', 'roll_max_1D', 'roll_min_1D', 'roll_mean_7D', 'roll_std_7D', 'roll_max_7D', 'roll_min_7D']

Linear numeric (21), categorical: ['day_of_week', 'hour']

Polynomial core (7): ['time_of_day', 'is_weekend', 'lag_1D', 'lag_7D', 'roll_mean_3h', 'roll_mean_1D', 'roll_mean_7D']
